# Naive Bayes Classification #

$Author$: Michael Simons

$Class$: MTH 448 - Data Oriented Computing

$Date$: 04/21/23

## Introduction ##

This report delves into the analysis and study of two datasets to test the effectiveness of the Naive Bayes classifier in making predictions. The first dataset comprises of a presidential debate transcript between Donald Trump and Hillary Clinton from September of 2016. The objective of this report is to evaluate the performance of the Naive Bayes classifier in predicting the speaker of a given sentence from this debate. This will provide insight towards the effectiveness of the classification technique.

## Scraping Debate Transcripts ##

The presidential debate which took place on September 26, 2016 between Clinton and Trump will be utilized to study how well a naive Bayes classifier can predict which person is speaking. To accomplish this task, the BeautifulSoup library will be used to parse the transcript in the form of an HTML file and the Pandas library will be used to store the results of the speaker's words counts.

We will start by opening the transcript file containing the debate transcript and fitting it into BeautifulSoup, as previously mentioned. Then we must convert the entire transcript to lowercase and remove filler tokens and punctuation such as exclamation marks, question marks, quotation marks etc. Next, a colon will be added to each speaker's name in the transcript and initialize a list of speaker tags. Finally, we will iterate through each word in the transcript and use counters to track the number of times each speaker said certain words. The result will be a Pandas DataFrame containing each speaker's word counts.

In [1]:
import pandas as pd
from bs4 import BeautifulSoup
from collections import Counter
import numpy as np

transcript_file = '6_September 26, 2016 Debate Transcript.html'

with open(transcript_file,encoding='utf-8') as f:            #Open file containing presidential debate transcript  
    soup = BeautifulSoup(f, 'html.parser')                   #Fit the transcript into BeautifulSoup
    
transcript = soup.text.lower().replace('—',' ')               #Make every letter lowercase,replace hyphens with space
tokens_to_remove=['--','[applause]','[inaudible]','[laughter]','[crosstalk]',',','.',';',':','!','?','"','(',')',"'"] #Filler tokens
for token in tokens_to_remove:                               #Iterate through filler tokens
    transcript = transcript.replace(token,'')                #Remove every intended token
    
    
list_of_speakers = ['holt','clinton','trump']
for speaker in list_of_speakers:                             #Iterate through the speakers
    transcript = transcript.replace(speaker,speaker+':')     #Add a colon to each speaker's name in the transcript
tags = [speaker+':' for speaker in list_of_speakers]         #Initialize list of speaker delimeters

holt = []                    #List of holt's words
clinton = []                 #List of clinton's words
trump = []                   #List of trump's words
uncertain=[]                 #List of words prior to a speaker's name appearing
current=uncertain            #Keep track of who is the current speaker

for word in transcript.split(): #Iterate through each word in the transcript.
    if  word == tags[0]:        #If holt's tag appears, set the current speaker to holt
        current = holt 
    elif word == tags[1]:       #If clinton's tag appears, set the current speaker to clinton
        current = clinton
    elif word == tags[2]:       #If trump's tag appears, set the current speaker to trump
        current = trump
    else:                       #Add the word to the current speaker's list of words
        current.append(word) 

#Use counters to track the amount of times each speaker said certain words. Initilize dataframe containing each counter.
dframe=pd.DataFrame({"holt":Counter(holt),"clinton":Counter(clinton),"trump":Counter(trump)}).fillna(0).reset_index()
dframe[:5] #Display first few entries to ensure proper formatting.

,index,holt,clinton,trump
0,nbc,1.0,0.0,0.0
1,news,1.0,0.0,0.0
2,good,2.0,20.0,18.0
3,evening,3.0,1.0,0.0
4,from,1.0,22.0,20.0


## Speaker Prediction ##

The data frame computed from the debate transcript will be used to develop a function called predict_speaker, which will predict the speaker given a fragment of text from the debate. The function takes this text fragment as input and returns a numpy array containing the log probability of the fragment being spoken by each speaker. The log probability will allow us to perform arithmetic opertations without underflow. This will avoid numerical issues.

The function works by first calculating the prior probabilities of each speaker, which are the proportions of words spoken by each speaker in the entire debate transcript. Then, for each word in the fragment, the function calculates the conditional probabilities of each speaker saying that word, which are the proportions of times that word was spoken by each speaker in the entire debate transcript. The product of the prior and conditional probabilities for each speaker is calculated, and the logarithm of the resulting array is returned.

In [2]:
total_words_spoken= np.array(dframe[["holt","clinton","trump"]].sum()).sum() #Count total words

words_spoken_by_holt = int(dframe[["holt"]].sum())                           #Total number of words spoken by holt
holt_component = words_spoken_by_holt/total_words_spoken                     #Pct of words spoken by holt

words_spoken_by_clinton = int(dframe[["clinton"]].sum())                     #Total number of words spoken by clinton
clinton_component = words_spoken_by_clinton/total_words_spoken               #Pct of words spoken by clinton

words_spoken_by_trump = int(dframe[["trump"]].sum())                         #Total number of words spoken by trump
trump_component = words_spoken_by_trump/total_words_spoken                   #Pct of words spoken by trump


#Takes a transcript fragment as an argument, and predicts whether holt,clinton, or trump is the speaker.
def predict_speaker(transcript_fragment):
    #Initialize array of the log of each speaker's component
    product=np.log(np.array([holt_component,clinton_component,trump_component]))
    for token in (['trump:','trump','clinton:','clinton','holt:','holt']+tokens_to_remove):  #Iterate through filler tokens
        transcript_fragment = transcript_fragment.lower().replace(token,'')  #Remove every intended token, lowercase each letter
    for word in transcript_fragment.split():                                 #Iterate through each word in the transcript
        w = dframe[dframe["index"]==word][["holt","clinton","trump"]]        #Grab the data frame at each word
        p = w+1/np.array(dframe[["holt","clinton","trump"]].sum())           #Compute each component to add
        product = product+np.log(np.array(p).flatten())                      #Take the log of each component, add to return val
    return list_of_speakers[product.argmax()] #Predict the speaker with the highest of the log probabilities

To test the predict_speaker function, a set of transcript fragments from the debate will passed to the function and the accuracy of  be and compare the predicted speaker to the actual speaker for each transcript fragment. We will select a set of transcript fragments, and pass each fragment to the predict_speaker function. This will allow us to compare the predicted speaker to the actual speaker, and get a sense of the function's accuracy. Since the predict_speaker function removes speaker tags in preprocessing, they can be included in the set to keep track of the actual speaker.

In [3]:
script_fragments = [
    "CLINTON: And I think it’s important that we grip this and deal with it, both at home and abroad. And here’s what we can do. We can deploy a half a billion more solar panels. We can have enough clean energy to power every home. We can build a new modern electric grid. That’s a lot of jobs; that’s a lot of new economic activity.",
    "HOLT: Mr. Trump?",
    "TRUMP: Wrong. Wrong.",
    "HOLT: Secretary Clinton, would you like to respond?",
    "TRUMP: She talks about solar panels. We invested in a solar company, our country. That was a disaster. They lost plenty of money on that one.",
    "CLINTON: Well, actually, I have thought about this quite a bit.",
    "TRUMP: Yeah, for 30 years.",
    "CLINTON: Well, I’ve been a senator, Donald…",
    "TRUMP: No, you’re wrong. You’re wrong.",
    "CLINTON: Well, that is just not accurate. I was against it once it was finally negotiated and the terms were laid out. I wrote about that in…",
    "CLINTON: I kind of assumed that there would be a lot of these charges and claims, and so…",
    "HOLT: Secretary Clinton, thank you.",
    "TRUMP: I’d just like to respond.",
    "TRUMP: It’s not negotiable, no. Let her release the e-mails. Why did she delete 33,000…",
    "HOLT: Mr. Trump, you have two minutes and the same question. Who’s behind it? And how do we fight it?",
    "TRUMP: Well, I’m really calling for major jobs, because the wealthy are going to create tremendous jobs. They’re going to expand their companies. They’re going to do a tremendous job."
]
correct_classifications=0      #Track numbers of successful classifications
attempts=0                     #Track numbers of attempted classifications
incorrects=[]                  #Track who was the actual speaker during incorrect classifications
for script_fragment in script_fragments:
    actual_speaker = script_fragment.split(':')[0].lower()      #Compute the actual speaker
    predicted_speaker = predict_speaker(script_fragment)        #Compute the predicted speaker
    if(actual_speaker==predicted_speaker):
        correct_classifications+=1                              #Add to correct attempts if correct
    else:
        incorrects.append(actual_speaker)                       #Track who was actual if incorrect
    attempts+=1    #Increment attempts
print(f"Classification Accuracy Over {len(script_fragments)} Transcript Fragments: {float(correct_classifications)/float(attempts)}")
print(f"The {len(incorrects)} times the classifier was incorrect, the actual speaker was: {incorrects}")

Classification Accuracy Over 16 Transcript Fragments: 0.8125
The 3 times the classifier was incorrect, the actual speaker was: ['holt', 'holt', 'holt']


The experiment found that out of the 16 transcript fragments, the predict_speaker function correctly identified the speaker in 13 cases, resulting in an accuracy of 81.25%. In each of the three cases which the function incorrectly identified the speaker, the actual speaker was debate moderator Holt. This is almost certainly due to Holt's drastically smaller sample size in comparison to the candidates who were speaking almost the entire time. These results demonstrate the effectiveness of the predict_speaker function, but also highlight the need for further refinement to improve accuracy.

## Conclusion ##


The performance of the classifier was evaluated, and it proved to be reliable, with the exception of agents such as Holt having a small sample size. The impact of different types of stopwords on the classifier's performance may be studied further. Furthermore, this project only considered one dataset, and it would be useful to apply this classifier to a variety of texts, such as social media posts or medical records. These avenues of research should lead to a deeper understanding of the strengths and limitations of the naive Bayes classifier for text classification tasks.

## References ##

[1] Naive Bayes Classifier," Towards Data Science, Jul. 11, 2019. https://towardsdatascience.com/naive-bayes-classifier-81d512f50a7c

[2] "September 26, 2016 Debate Transcript," Debates, 2016. https://www.debates.org/voter-education/debate-transcripts/september-26-2016-debate-transcript/